In [ ]:

from dataclasses import dataclass
@dataclass
class Instruction:
    name: str
    destination: str
    sources: list
# Instruction set untuk menguji RAW, WAR, dan WAW
instructions = [
    Instruction("I1", "R1", ["R1", "R3"]),
    Instruction("I2", "R4", ["R1", "R4"]),
    Instruction("I3", "R1", ["R6", "R7"]),
    Instruction("I4", "R4", ["R8", "R9"]),
    Instruction("I5", "R6", ["R4", "R10"]),
    Instruction("I6", "R11", ["R1", "R4"]),
]

In [ ]:
def dependency_types(inst1, inst2):
    """Mendeteksi semua dependensi dari inst1 ke inst2."""
    dependencies = []

    # RAW: inst2 membaca register yang ditulis inst1.
    if inst1.destination in inst2.sources:
        dependencies.append(("RAW", inst1.destination))

    # WAR: inst2 menulis register yang dibaca inst1.
    if inst2.destination in inst1.sources:
        dependencies.append(("WAR", inst2.destination))

    # WAW: kedua instruksi menulis register tujuan yang sama.
    if inst1.destination == inst2.destination:
        dependencies.append(("WAW", inst1.destination))

    return dependencies

In [ ]:
def build_dependency_graph(instructions):
    """Membentuk sisi graph sesuai urutan instruksi."""
    edges = []

    for i in range(len(instructions)):
        for j in range(i + 1, len(instructions)):
            dependencies = dependency_types(
                instructions[i], instructions[j]
            )

            if dependencies:
                edges.append((
                    instructions[i].name,
                    instructions[j].name,
                    dependencies
                ))

    return edges

In [ ]:
def show_dependencies(instructions):
    print("\n" + "=" * 60)
    print("DEPENDENCY ANALYSIS")
    print("=" * 60)

    edges = build_dependency_graph(instructions)

    for source, target, dependencies in edges:
        details = ", ".join(
            f"{kind} ({register})"
            for kind, register in dependencies
        )
        print(f"{source} -> {target}: {details}")

    return edges

In [ ]:
def sequential_execution(instructions):
    print("\n" + "=" * 60)
    print("SEQUENTIAL EXECUTION")
    print("=" * 60)

    for cycle, inst in enumerate(instructions, start=1):
        print(
            f"Cycle {cycle}: {inst.name}: "
            f"{inst.destination} = {' + '.join(inst.sources)}"
        )

    print(f"Total Sequential Cycles: {len(instructions)}")
    return len(instructions)


def parallel_execution(instructions, edges):
    print("\n" + "=" * 60)
    print("INSTRUCTION-LEVEL PARALLEL EXECUTION")
    print("=" * 60)

    predecessors = {inst.name: set() for inst in instructions}

    for source, target, _ in edges:
        predecessors[target].add(source)

    completed = set()
    remaining = instructions.copy()
    cycle = 0

    while remaining:
        cycle += 1

        ready = [
            inst for inst in remaining
            if predecessors[inst.name].issubset(completed)
        ]

        if not ready:
            raise RuntimeError("Tidak ditemukan instruksi yang siap.")

        print(f"\nCycle {cycle}:")

        for inst in ready:
            print(
                f"  {inst.name}: {inst.destination} = "
                f"{' + '.join(inst.sources)}"
            )

        for inst in ready:
            completed.add(inst.name)
            remaining.remove(inst)

    print(f"Total ILP Cycles: {cycle}")
    return cycle

In [ ]:
def main():
    print("=" * 60)
    print("INSTRUCTION-LEVEL PARALLELISM SIMULATOR")
    print("=" * 60)

    print("\nInstruction Set:")
    for inst in instructions:
        print(
            f"{inst.name}: {inst.destination} = "
            f"{' + '.join(inst.sources)}"
        )

    edges = show_dependencies(instructions)

    sequential_cycles = sequential_execution(instructions)
    parallel_cycles = parallel_execution(instructions, edges)

    speedup = sequential_cycles / parallel_cycles
    efficiency = speedup / len(instructions) * 100

    print("\n" + "=" * 60)
    print("PERFORMANCE COMPARISON")
    print("=" * 60)
    print(f"Sequential Cycles : {sequential_cycles}")
    print(f"ILP Cycles        : {parallel_cycles}")
    print(f"Speedup           : {speedup:.2f}x")
    print(f"ILP Efficiency    : {efficiency:.2f}%")

    print("\nDEPENDENCY GRAPH")
    for source, target, dependencies in edges:
        labels = "/".join(kind for kind, _ in dependencies)
        print(f"{source} ---> {target} [{labels}]")


if __name__ == "__main__":
    main()